# PCA of OxCGRT custom policy fields

Principal component analysis of the nine OxCGRT indicators used in the `custom` field set, for a single country.
The main output is the share of variance captured by each component.

## The approach

Treat each day as an observation and each policy indicator as a variable. This creates a matrix with one row per day and nine columns, each already scaled to $[0, 1]$.

PCA finds orthogonal linear combinations of those nine series that successively capture as much of the day-to-day variation as possible. If many restrictions were tightened and loosened together, PC1 will look like a general stringency factor and will explain a large fraction of the variance. Later components then describe residual patterns (for example face coverings moving independently of workplace closing). That is useful context for the weighted-policy model: highly collinear inputs are hard to separate into independent weights.

## Preprocessing

There is less to do than for a typical PCA, because the series are already on a common scale, but a few choices still matter. `scale_oxcgrt_pols` divides each indicator by its OxCGRT maximum, so every series is in $[0, 1]$. PCA is defined on covariances, so each column has its mean subtracted. A policy that sat at 1.0 for the whole window contributes nothing after centering. `sklearn.decomposition.PCA` centres automatically.

The other common step is dividing each column by its standard deviation. That forces every policy to contribute equally, even one that barely moved. Because the indicators are already on $[0, 1]$, a policy that actually swung from 0 to 1 should weigh more than one that stayed put. We therefore run covariance PCA on the max-normalised values.

The time window is the country's OxCGRT simulation period from `find_run_start_time` to `find_run_end_time`. Drop any day with a missing value on any of the nine fields. National OxCGRT is usually complete; if a country is not, proceed to deleting that row. The calibration smooths with a 7-day rolling mean, which is not done here.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from emu_renewal.constants import OXCGRT_COLMAP, OXCGRT_LOCS
from emu_renewal.inputs import (
    find_oxcgrt_country_data,
    get_country_pop,
    get_oxcgrt_data,
    get_rel_oxcgrt_cols,
    scale_oxcgrt_pols,
)
from emu_renewal.run import find_run_end_time, find_run_start_time
from emu_renewal.utils import get_country_name

In [ ]:
iso3 = "SWE"
start = find_run_start_time(get_country_pop(iso3), iso3)
end = find_run_end_time(iso3, "oxcgrt")
get_country_name(iso3), pd.Timestamp(start).date(), pd.Timestamp(end).date()

In [ ]:
pol = find_oxcgrt_country_data(iso3, get_oxcgrt_data())
scaled = scale_oxcgrt_pols(pol[get_rel_oxcgrt_cols("M", pol)])
policies = scaled[OXCGRT_COLMAP["custom"]].rename(columns=OXCGRT_LOCS)

policies = policies.loc[
    (policies.index >= pd.Timestamp(start)) & (policies.index <= pd.Timestamp(end))
]

pca_matrix = policies.dropna()
dropped = policies.shape[0] - pca_matrix.shape[0]
print(f"{pca_matrix.shape[0]} days, {pca_matrix.shape[1]} policies ({dropped} days dropped for missing values)")
print(f"{pca_matrix.index.min().date()} to {pca_matrix.index.max().date()}")

In [ ]:
# Check column means and standard deviations
pca_matrix.agg(["mean", "std", "min", "max"]).T

In [ ]:
pca = PCA()
pca.fit(pca_matrix)

variance = pd.DataFrame(
    {
        "variance_share": pca.explained_variance_ratio_,
        "cumulative": pca.explained_variance_ratio_.cumsum(),
    },
    index=[f"PC{i + 1}" for i in range(pca.n_components_)],
)
variance

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(variance.index, variance["variance_share"], color="steelblue")
ax.set_ylabel("Share of variance")
ax.set_ylim(0, 1)

ax2 = ax.twinx()
ax2.plot(variance.index, variance["cumulative"], color="black", marker="o")
ax2.set_ylabel("Cumulative share")
ax2.set_ylim(0, 1.05)

ax.set_title(f"OxCGRT custom fields, {get_country_name(iso3)}")
fig.tight_layout()
plt.show()

Loadings are the weights of each policy on each component (rows of `pca.components_`, transposed). PC1 loadings with the same sign mean those policies moved together. This is extra to the variance shares, but it is the usual next thing to look at.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=pca_matrix.columns,
    columns=variance.index,
)
loadings.round(2)

Because each principal component is a unit-length vector, the square of a policy’s loading on PC1 is that policy’s share of the component. Ranking by absolute loading (equivalently by this share) picks out which restrictions actually moved along the main axis of variation.

Stop at two or three. The eigenvector mixes every policy a little, and by the fourth the same series often loads as strongly on PC2, so listing more starts to describe a different direction. The sign of the loading still matters: policies with the same sign tightened and loosened together. (sklearn can flip the whole of PC1; only relative signs are meaningful.)


In [ ]:
N_PC1_POLICIES = 3
pc1 = loadings["PC1"]
ranked = pd.DataFrame(
    {
        "PC1 loading": pc1,
        "share of PC1": pc1.pow(2),
        "PC2 loading": loadings["PC2"],
    }
).sort_values("share of PC1", ascending=False)

top = ranked.head(N_PC1_POLICIES)
print(top.round(2).to_string())

Check for similar results after z-scoring each policy (`StandardScaler`). Variance shares will differ if some indicators moved much more than others.

In [ ]:
pca_z = PCA().fit(StandardScaler().fit_transform(pca_matrix))
var_z = pd.DataFrame(
    {
        "variance_share": pca_z.explained_variance_ratio_,
        "cumulative": pca_z.explained_variance_ratio_.cumsum(),
    },
    index=[f"PC{i + 1}" for i in range(pca_z.n_components_)],
)
var_z